# Etapa 1: Introdução e Análise Inicial do Sinal ECG (MIT BIH)
## Projeto de Processamento de Sinais Digitais (Grupo 6)

> Pedro Arthur, RA: 814248

> Juliano Eleno Silva Pádua, RA: 800812

> Matheo, RA: 821293

**Registo de referência:** MIT BIH 100 (ritmo sinusal normal)

* Esta apresentação cobre a Etapa 01 do projeto.
* Objetivo: remover ruídos típicos de ECG.
* Validação estritamente visual sobre os 10 segundos iniciais do Registo 100.

## 1. Resumo do Projeto e Objetivos

**Problema.** Sinais reais de ECG carregam três famílias de ruído que comprometem o diagnóstico:

* Baseline wander (abaixo de 0.5 Hz), associado à respiração e ao movimento do paciente.
* Interferência da rede elétrica (PLI), próxima de 60 Hz, decorrente do acoplamento à rede de energia.
* Ruído muscular ou EMG (acima de 40 Hz), originado por contrações musculares involuntárias.

**Objetivos da Fase 1.**

* Atenuar as três famílias de ruído com filtros FIR de fase linear.
* Preservar a morfologia das ondas P, do complexo QRS e da onda T, evitando deslocamento temporal.
* Realizar uma comparação visual, no domínio do tempo, entre dois métodos de projeto FIR aplicados ao sinal de referência.

## 2. Anatomia do ECG: Ondas P, QRS e T

![Anatomia do ECG](../imgs/image.png)

Cada batimento cardíaco gera um padrão fisiológico bem definido no ECG:

* Onda P: despolarização (ativação elétrica) dos átrios. Pequena amplitude e baixa frequência.
* Complexo QRS: despolarização dos ventrículos. É o pico de maior amplitude e a marca temporal do batimento (pico R).
* Onda T: repolarização ventricular, ou seja, a recuperação elétrica do coração antes do próximo ciclo.

**Justificativa do filtro FIR de fase linear.**

* Atraso de grupo constante: todos os componentes de frequência são deslocados pelo mesmo tempo.
* Resultado: as ondas P, QRS e T não são deformadas nem deslocadas relativamente entre si.
* Trata se de uma exigência clínica, pois medir intervalos PR, QRS e QT requer preservação morfológica.

## 3. Base de Dados: MIT BIH Arrhythmia Database

* Referência mundial em cardiologia computacional, distribuída pela PhysioNet.
* 48 registos de ECG de pacientes reais, com aproximadamente 30 minutos cada e dois canais (tipicamente MLII e V5).
* Frequência de amostragem: `fs = 360 Hz` (Nyquist em 180 Hz, suficiente para a faixa diagnóstica entre 0.5 Hz e 40 Hz).
* Anotações `.atr` produzidas por especialistas, identificando o instante do pico R e o tipo de batimento.
* Registo 100: escolhido para a Fase 1 por apresentar ritmo sinusal normal, servindo de referência de normalidade para validar que o filtro não introduz artefatos.

In [ ]:
from __future__ import annotations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import wfdb

from src.config import mitdb_record_dir
from src.preprocessing.fir_filters import (
    design_highpass,
    design_highpass_remez,
    design_bandstop,
    design_bandstop_remez,
    design_lowpass,
    design_lowpass_remez,
    apply_filtfilt,
)

RECORD_NAME = "100"
record_path = (Path(mitdb_record_dir()) / RECORD_NAME).as_posix()

record = wfdb.rdrecord(record_path)
ann = wfdb.rdann(record_path, "atr")

print("fs (Hz):", record.fs)
print("Canais:", record.sig_name)
print("Duração (s):", record.sig_len / record.fs)

## 4. Visualização do Sinal Cru: Registo 100, primeiros 10 s

* Janela de 10 segundos (`fs · 10 = 3600 amostras`) no canal MLII (derivação modificada II).
* As linhas verticais marcam as anotações WFDB dos picos R fornecidas pelos cardiologistas.
* Observa se a oscilação de baixa frequência da linha de base e o ruído fino sobreposto ao traçado.

In [ ]:
fs = float(record.fs)
window = int(fs * 10)
t = np.arange(window) / fs

ch = 0  # MLII
x_raw = record.p_signal[:window, ch]
ann_in_window = ann.sample[ann.sample < window]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, x_raw, color="tab:blue", linewidth=0.9, label="ECG cru (MLII)")
for s in ann_in_window:
    ax.axvline(s / fs, color="tab:red", alpha=0.3, linewidth=0.8)
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Amplitude (mV)")
ax.set_title(f"MIT BIH {RECORD_NAME}: primeiros 10 s, canal {record.sig_name[ch]}")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## 5. Estratégia de Filtragem: FIR de Fase Linear

A Fase 1 utiliza uma cascata de três filtros FIR projetados sobre `fs = 360 Hz`:

1. Passa alta com corte em 0.5 Hz, voltado ao baseline wander (abaixo de 0.5 Hz).
2. Rejeita faixa entre 59 Hz e 61 Hz, voltado à interferência da rede elétrica (PLI em 60 Hz).
3. Passa baixa com corte em 40 Hz, voltado ao ruído muscular (acima de 40 Hz).

São comparadas duas técnicas de projeto:

* Janela de Hamming: método clássico, simples e robusto, com ordem aproximada de `3.3 · fs / Δf`.
* Parks McClellan (Remez): projeto equiripple ótimo no sentido de Chebyshev, alcançando ordens menores para uma mesma especificação.

A aplicação ocorre em fase zero por meio de `filtfilt` (forward backward), garantindo atraso nulo e preservação da posição temporal dos picos R.

In [ ]:
# Pipeline FIR projetado pela janela de Hamming
h_hp_ham = design_highpass(fs=fs, cutoff_hz=0.5, transition_hz=0.5)
h_bs_ham = design_bandstop(fs=fs, low_hz=59.0, high_hz=61.0, transition_hz=1.0)
h_lp_ham = design_lowpass(fs=fs, cutoff_hz=40.0, transition_hz=8.0)

# Pipeline FIR projetado por Parks McClellan (equiripple)
h_hp_pm = design_highpass_remez(fs=fs, cutoff_hz=0.5, transition_hz=0.5)
h_bs_pm = design_bandstop_remez(fs=fs, low_hz=59.0, high_hz=61.0, transition_hz=1.0)
h_lp_pm = design_lowpass_remez(fs=fs, cutoff_hz=40.0, transition_hz=8.0)

print(f"Hamming         | HP: N = {len(h_hp_ham):4d} | BS: N = {len(h_bs_ham):4d} | LP: N = {len(h_lp_ham):4d}")
print(f"Parks McClellan | HP: N = {len(h_hp_pm):4d} | BS: N = {len(h_bs_pm):4d} | LP: N = {len(h_lp_pm):4d}")

In [ ]:
# Aplicação completa em fase zero (filtfilt) sobre o registo inteiro,
# com posterior recorte para os 10 s do canal MLII. Filtrar o registo
# completo evita problemas com padlen do filtfilt para FIRs longos e
# elimina transientes de borda dentro da janela visualizada.
x_full = record.p_signal[:, ch]

# Cascata Hamming: HP, BS, LP
xh = apply_filtfilt(h_hp_ham, x_full)
xh = apply_filtfilt(h_bs_ham, xh)
xh = apply_filtfilt(h_lp_ham, xh)
x_filt_ham = xh[:window]

# Cascata Parks McClellan: HP, BS, LP
xp = apply_filtfilt(h_hp_pm, x_full)
xp = apply_filtfilt(h_bs_pm, xp)
xp = apply_filtfilt(h_lp_pm, xp)
x_filt_pm = xp[:window]

## 6. Resultados: Comparação Visual no Domínio do Tempo

A avaliação desta etapa é estritamente visual e contempla três representações sobre a mesma janela de 10 segundos do canal MLII:

1. Sinal original (cru), antes de qualquer processamento.
2. Sinal processado pelo pipeline FIR projetado por janela de Hamming.
3. Sinal processado pelo pipeline FIR projetado por Parks McClellan.

Espera se que ambos os pipelines estabilizem a linha de base, atenuem o tremor de alta frequência e preservem a posição e a amplitude relativa das ondas P, do complexo QRS e da onda T.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, sharey=True)

axes[0].plot(t, x_raw, color="tab:blue", linewidth=0.9, label="Original")
axes[0].set_title("Sinal original (cru)")
axes[0].set_ylabel("Amplitude (mV)")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="upper right")

axes[1].plot(t, x_raw, color="gray", alpha=0.5, linewidth=0.8, label="Original")
axes[1].plot(t, x_filt_ham, color="tab:orange", linewidth=1.0, label="Filtrado")
axes[1].set_title("FIR via janela de Hamming")
axes[1].set_ylabel("Amplitude (mV)")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="upper right")

axes[2].plot(t, x_raw, color="gray", alpha=0.5, linewidth=0.8, label="Original")
axes[2].plot(t, x_filt_pm, color="tab:green", linewidth=1.0, label="Filtrado")
axes[2].set_title("FIR via Parks McClellan (equiripple)")
axes[2].set_xlabel("Tempo (s)")
axes[2].set_ylabel("Amplitude (mV)")
axes[2].grid(True, alpha=0.3)
axes[2].legend(loc="upper right")

fig.suptitle(f"MIT BIH {RECORD_NAME}: comparacao visual no dominio do tempo")
fig.tight_layout()
plt.show()

## 7. Cronograma e Próximas Etapas

Com a Fase 1 (28/04) concluída, consolidando o fluxo de denoising e a preservação morfológica fundamental por meio do filtro FIR, o projeto avança seguindo o cronograma oficial da disciplina:

* Fase 2 (Vídeo 2 em 02/06): Implementação e Comparação de Técnicas
    1. Detalhamento e explicação aprofundada das técnicas base.
    2. Implementação de outros filtros estruturais para fins de comparação com o FIR original.
    3. Evolução para a análise de tempo-frequência: uso de espectrogramas (STFT) e filtros de Gabor 1D (janelas gaussianas) para acompanhar variações e caracterizar batimentos anômalos.
* Entrega do Projeto (08/07): Consolidação
    1. Entrega do artefato final em código (notebook completo com todas as execuções).
    2. Submissão do artigo redigido em LaTeX documentando os resultados e validações.
* Apresentação Final (08, 14 e 15/07)
    1. Defesa do projeto completo (20 a 30 minutos de apresentação e 10 minutos de perguntas).